In [1]:
# ==============================================================================
# PIPELINE 1: Data Preparation & Feature Extraction
# ==============================================================================
import pandas as pd
from src.preparation.ingestion import convert_csv_to_parquet
from src.profiling import generate_minimal_report, generate_full_sample_report
from src.preparation.reduction import filter_and_reduce
from src.preparation.formatting import (
    convert_yes_no_to_binary, 
    preprocess_framework_estimation, 
    standardize_categorical_features, 
    create_one_hot_encodings
)
from src.preparation.consolidation import (
    consolidate_cascade_feature, 
    consolidate_logical_or_feature, 
    consolidate_tenders_and_non_awards
)
from src.preparation.extraction import (
    calculate_days, 
    reduce_cpv_cardinality, 
    extract_nuts_features, 
    extract_winner_features
)

pd.set_option('display.max_columns', None)

In [2]:
# ------------------------------------------------------------------------------
# STEP 1: Data Ingestion (Raw CSV -> Parquet)
# ------------------------------------------------------------------------------

RUN_INGESTION = False

if RUN_INGESTION:
    df_cfc = convert_csv_to_parquet(
        csv_path="data/raw/export_CFC_2018_2023.csv",
        parquet_path="data/raw/CFC_2018_2023.parquet",
        dataset_name="CFC (Contract Notices)"
    )
    df_can = convert_csv_to_parquet(
        csv_path="data/raw/export_CAN_2018_2023.csv",
        parquet_path="data/raw/CAN_2018_2023.parquet",
        dataset_name="CAN (Contract Award Notices)"
    )
else:
    df_cfc = pd.read_parquet("data/raw/CFC_2018_2023.parquet")
    df_can = pd.read_parquet("data/raw/CAN_2018_2023.parquet")



In [3]:
# ------------------------------------------------------------------------------
# STEP 2: Automated Profiling (Data Understanding on RAW data)
# ------------------------------------------------------------------------------

# Generate a minimal report for the full CFC data
generate_minimal_report(
    df=df_cfc,
    output_path="reports/raw/cfc_minimal.html",
    title="CFC Raw (minimal; full data)"
)

# Generate a full report for a sample of the CFC data
generate_full_sample_report(
    df=df_cfc,
    output_path="reports/raw/cfc_full_sample.html",
    title="CFC Raw (full; 100k sample)",
    sample_size=100000
)

# Generate a minimal report for the full CAN data
generate_minimal_report(
    df=df_can,
    output_path="reports/raw/can_minimal.html",
    title="CAN Raw (minimal; full data)"
)

# Generate a full report for a sample of thew CAN data
generate_full_sample_report(
    df=df_can,
    output_path="reports/raw/can_full_sample.html",
    title="CAN Raw (full; 100k sample)",
    sample_size=100000
)

Generating minimal report: 'CFC Raw (minimal; full data)' ...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 64/64 [00:19<00:00,  3.22it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 75/75 [00:13<00:00,  5.49it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> Full report successfully saved to: reports/raw/can_full_sample.html



In [4]:
# ------------------------------------------------------------------------------
# STEP 3: Reduction (Filtering & Dropping)
# ------------------------------------------------------------------------------

cols_to_drop_both = [
    "ISO_COUNTRY_CODE_ALL", # high NaNs
    "EU_INST_CODE", # relevance; high NaNs
    "B_ACCELERATED", # high NaNs
    "B_NUMBER_COUNTRIES", # skewed
    "B_MULTIPLE_CAE", # relevance (-> COOPERATIVE_PURCHASING)
    "TED_NOTICE_URL", # relevance
    "CAE_NAME", # relevance (-> ISO_COUNTRY_CODE)
    "CAE_ADDRESS", # relevance (-> ISO_COUNTRY_CODE)
    "CAE_TOWN", # relevance (-> ISO_COUNTRY_CODE)
    "CAE_POSTAL_CODE", # relevance (-> ISO_COUNTRY_CODE)
    "CAE_NATIONALID", # relevance; high NaNs (-> ISO_COUNTRY_CODE)
    "B_GPA", # relevance
    "GPA_COVERAGE", # relevance; high NaNs
    "CAE_GPA_ANNEX", # relevance; high NaNs
    "ISO_COUNTRY_CODE_GPA", # relevance; high NaNs
    "MAIN_CPV_CODE_GPA", # relevance; high NaNs
    "B_ELECTRONIC_AUCTION", # relevance; skewed
    "CRIT_CRITERIA", # relevance; high NaNs
    "CRIT_WEIGHTS", # high NaNs
    "CRIT_PRICE_WEIGHT", # high NaNs
    "CANCELLED", # skewed
    "CORRECTIONS", # relevance
    "B_DYN_PURCH_SYST", # relevance
    "XSD_VERSION", # relevance
    "ADDITIONAL_CPVS" # relevance
]

# CFC specific columns to drop
cols_to_drop_cfc = cols_to_drop_both + [
    "ENV_OPERATORS", # high NaNs
    "ENV_MIN_OPERATORS", # high NaNs
    "ENV_MAX_OPERATORS", # high NaNs
    "B_LANGUAGE_ANY_EC", # high NaNs
    "ADMIN_LANGUAGES_TENDER", # relevance
    "ADMIN_OTHER_LANGUAGES_TENDER", # high NaNs
    "LOTS_SUBMISSION", # high NaNs
    "FUTURE_CAN_ID_ESTIMATED", # unreliable
    "B_VARIANTS", # relevance
    "B_OPTIONS", # relevance
    "B_RENEWALS", # relevance
    "B_FRA_SINGLE_OPERATOR", # relevance
    "FRA_NUMBER_OPERATORS", # high NaNs
    "FRA_NUMBER_MAX_OPERATORS", # high NaNs
    "CONTRACT_START", # high NaNs
    "CONTRACT_COMPLETION" # high NaNs
]

# CAN specific columns to drop
cols_to_drop_can = cols_to_drop_both + [
    "WIN_NAME", # relevance
    "WIN_ADDRESS", # relevance
    "WIN_TOWN", # relevance
    "WIN_POSTAL_CODE", # relevance
    "WIN_NATIONALID", # relevance
    "TITLE", # relevance
    "CONTRACT_NUMBER", # relevance
    "OUT_OF_DIRECTIVES", # relevance
    "INFO_UNPUBLISHED", # relevance
    "B_SUBCONTRACTED", # relevance
    "B_AWARDED_TO_A_GROUP", # relevance
    "NUMBER_OFFERS_ELECTR", # relevance
    "ID_AWARD", # relevance
    "ID_LOT_AWARED" # relevance (-> ID_LOT)
]

# Execute Cleaning
df_cfc_clean = filter_and_reduce(df_cfc, "CFC", cols_to_drop_cfc)
df_can_clean = filter_and_reduce(df_can, "CAN", cols_to_drop_can)

datasets = {'CFC': df_cfc_clean, 'CAN': df_can_clean}


 -> [CFC] Excluded 29,374 invalid records (e.g., cancelled, out of scope).
 -> [CFC] Dropped 36 unneeded columns to reduce dimensionality.
 -> [CAN] Excluded 169,015 invalid records (e.g., cancelled, out of scope).
 -> [CAN] Dropped 37 unneeded columns to reduce dimensionality.


In [5]:
# ------------------------------------------------------------------------------
# STEP 4: Formatting & Feature Consolidation
# ------------------------------------------------------------------------------
print("Starting Formatting & Consolidation...")

for name, df in datasets.items():
    print(f"\n--- Processing {name} ---")
    
    # 1. Base Formatting (Prepares fields for logical operations)
    df = convert_yes_no_to_binary(df, ['B_EU_FUNDS', 'B_RECURRENT_PROCUREMENT'])
    df = preprocess_framework_estimation(df)
    
    # 2. Price Consolidation
    df = consolidate_cascade_feature(df, target_col='TOTAL_VALUE_EUR', cascade_cols=['VALUE_EURO_FIN_2', 'VALUE_EURO_FIN_1', 'VALUE_EURO'])
    if name == 'CAN':
        df = consolidate_cascade_feature(df, target_col='LOT_AWARD_VALUE_EUR', cascade_cols=['AWARD_VALUE_EURO_FIN_1', 'AWARD_VALUE_EURO', 'AWARD_EST_VALUE_EURO'])
        
    # 3. Framework & Cooperative Purchasing Consolidation
    df = consolidate_logical_or_feature(df, target_col='IS_FRAMEWORK', source_cols=['B_FRA_AGREEMENT', 'B_FRA_ESTIMATED_FLAG', 'B_FRA_CONTRACT'])
    df = consolidate_logical_or_feature(df, target_col='COOPERATIVE_PURCHASING', source_cols=['B_ON_BEHALF', 'B_INVOLVES_JOINT_PROCUREMENT', 'B_AWARDED_BY_CENTRAL_BODY'])
    
    # 4. Target Consolidation (Only relevant for CAN)
    if name == 'CAN':
        df = consolidate_tenders_and_non_awards(df)

    # Save back to dictionary
    datasets[name] = df

Starting Formatting & Consolidation...

--- Processing CFC ---
 -> Converted features to binary (1/0) format: ['B_EU_FUNDS', 'B_RECURRENT_PROCUREMENT']
 -> Pre-processed algorithmic 'FRA_ESTIMATED' strings into binary flag.
 -> Consolidated cascade feature 'TOTAL_VALUE_EUR'. Valid values: 4,318,583
 -> Consolidated 'IS_FRAMEWORK' via logical OR. Total positive flags: 3,110,145
 -> Consolidated 'COOPERATIVE_PURCHASING' via logical OR. Total positive flags: 925,456

--- Processing CAN ---
 -> Converted features to binary (1/0) format: ['B_EU_FUNDS']
 -> Pre-processed algorithmic 'FRA_ESTIMATED' strings into binary flag.
 -> Consolidated cascade feature 'TOTAL_VALUE_EUR'. Valid values: 5,494,554
 -> Consolidated cascade feature 'LOT_AWARD_VALUE_EUR'. Valid values: 5,074,884
 -> Consolidated 'IS_FRAMEWORK' via logical OR. Total positive flags: 2,401,401
 -> Consolidated 'COOPERATIVE_PURCHASING' via logical OR. Total positive flags: 463,905
 -> Excluded 189,985 rows due to administrative ca

In [6]:
# ------------------------------------------------------------------------------
# STEP 5: Extraction & ML Encoding
# ------------------------------------------------------------------------------
print("Starting Extraction & Encoding...")
high_card_cols = ['ISO_COUNTRY_CODE', 'CAE_TYPE', 'TOP_TYPE', 'MAIN_ACTIVITY']

for name, df in datasets.items():
    print(f"\n--- Transforming {name} ---")
    
    # 1. Extraction (Parsing complex metrics)
    df = reduce_cpv_cardinality(df, digits=2)
    df = extract_nuts_features(df)
    
    if name == 'CFC':
        df = calculate_days(df, 'DT_DISPATCH', 'DT_APPLICATIONS', 'PREPARATION_DAYS')
    elif name == 'CAN':
        df = calculate_days(df, 'DT_DISPATCH', 'DT_AWARD', 'DAYS_TO_AWARD')
        df = extract_winner_features(df)

    # 2. Final ML Encodings (Categoricals & One-Hot)
    df = standardize_categorical_features(df, high_card_cols)
    df = create_one_hot_encodings(df)
    
    # Save back to dictionary
    datasets[name] = df

# Entpacken für das Speichern
df_cfc_final = datasets['CFC']
df_can_final = datasets['CAN']

Starting Extraction & Encoding...

--- Transforming CFC ---


/home/msuess/Documents/KIDS/Project/src/preparation/extraction.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['79' '79' '79' ... '79' '71' '71']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[valid_mask, 'CPV'] = cpv_cleaned


 -> Reduced CPV hierarchical cardinality: 7,790 -> 45 unique categories.
 -> Extracted spatial complexity metrics ('NUTS_REGION_COUNT', 'NUTS_LEVEL').
 -> Calculated 'PREPARATION_DAYS'. Valid numeric values: 7,680,335
 -> Standardized 4 features into categorical format.
 -> Generated 5 binary one-hot encoded features.

--- Transforming CAN ---


/home/msuess/Documents/KIDS/Project/src/preparation/extraction.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['72' '79' '79' ... '85' '85' '85']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[valid_mask, 'CPV'] = cpv_cleaned


 -> Reduced CPV hierarchical cardinality: 7,739 -> 45 unique categories.
 -> Extracted spatial complexity metrics ('NUTS_REGION_COUNT', 'NUTS_LEVEL').
 -> Calculated 'DAYS_TO_AWARD'. Valid numeric values: 4,139,277
 -> Extracted 3 quantitative market composition metrics from winner strings.
 -> Standardized 4 features into categorical format.
 -> Generated 5 binary one-hot encoded features.


In [7]:
# ------------------------------------------------------------------------------
# STEP 6: Final Output Checkpoint
# ------------------------------------------------------------------------------
df_cfc_final.to_parquet("data/prepared/CFC_2018_2023.parquet", index=False)
df_can_final.to_parquet("data/prepared/CAN_2018_2023.parquet", index=False)

# Generate a minimal report for the full CFC data
generate_minimal_report(
    df=df_cfc_final,
    output_path="reports/processed/cfc_minimal.html",
    title="CFC Processed (minimal; full data)"
)

# Generate a minimal report for the full CAN data
generate_minimal_report(
    df=df_can_final,
    output_path="reports/processed/can_minimal.html",
    title="CAN Processed (minimal; full data)"
)

Generating minimal report: 'CFC Processed (minimal; full data)' ...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 26/26 [00:02<00:00, 11.44it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 31/31 [00:02<00:00, 12.98it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> Minimal report successfully saved to: reports/processed/can_minimal.html

